In [1]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# 使用DBSCAN聚类分割
print('正在加载点云....')
pcd = o3d.io.read_point_cloud('../data/points/tutorials/room_scan1.pcd')
print(pcd)
o3d.visualization.draw_geometries([pcd])


正在加载点云....
PointCloud with 112586 points.
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: 不支持请求的转换操作。 


In [4]:
print('正在DBSCAN聚类')
eps = 0.5
min_points = 1000
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array(pcd.cluster_dbscan(eps, min_points, print_progress=True))
max_label = labels.max()
print(f'point cloud has {max_label + 1} clusters')
colors = plt.get_cmap('tab20')(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd])

正在DBSCAN聚类
[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 3
point cloud has 3 clusters


In [7]:
# RANSAC平面分割
distance_threshhold = 0.5
ransac_n = 3
num_iters = 1000

# RANSAC模型随机采样三个点确定一个平面，根据这三个点计算平面方程，存储在plane_model中
# 计算所有点到该平面的距离，统计距离小于阈值的内点数量
# 重复上述过程，选择内点最多的平面模型
plane_model, inliers = pcd.segment_plane(distance_threshhold, ransac_n, num_iters)

[a, b, c, d] = plane_model
print(f'Plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0')

inlier_cloud = pcd.select_by_index(inliers)
inlier_cloud.paint_uniform_color([0, 0, 1])
print(inlier_cloud)

outlier_cloud = pcd.select_by_index(inliers, invert=True)
outlier_cloud.paint_uniform_color([1, 0, 0])
print(outlier_cloud)

o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])

Plane equation: 0.91x + -0.38y + 0.16z + -0.14 = 0
PointCloud with 52270 points.
PointCloud with 60316 points.


In [7]:
# 隐藏点剔除
print('正在剔除隐藏点....')
#点云包围盒的最大角点坐标与最小角点坐标相减得到包围盒的尺寸向量，求范数得到对角线距离
diameter = np.linalg.norm(np.asarray(pcd.get_max_bound()) - np.asarray(pcd.get_min_bound()))
print('定义隐藏点去除的参数')

# 球形投影的半径，要足够大以包含整个点云
radius = diameter * 100
# 虚拟相机位置，放在点云正前方距离为diameter的位置
cameras = [
    [0, 0, diameter],        # 正面
    [diameter, 0, 0],        # 右侧
    [-diameter, 0, 0],       # 左侧
    [0, diameter, 0]         # 顶部
]

all_visible_indices = set()
for cam in cameras:
    # 将点云投影到一个以camera为中心、半径为radius的大球面上
    # 从相机视角出发，只有最前面的点（深度值最小）是可见的
    # 获取视点位置能够看见的所有索引点的位置
    _, indices = pcd.hidden_point_removal(cam, radius)
    all_visible_indices.update(indices)

# 可视化点云
pcd_visible = pcd.select_by_index(list(all_visible_indices))
pcd_visible.paint_uniform_color([0, 0, 1])
print('可视点个数为：', pcd_visible)

pcd_hidden = pcd.select_by_index(list(all_visible_indices), invert=True)
pcd_hidden.paint_uniform_color([1, 0, 0])
print('隐藏点个数为：', pcd_hidden)

o3d.visualization.draw_geometries([pcd_visible, pcd_hidden])

正在剔除隐藏点....
定义隐藏点去除的参数
可视点个数为： PointCloud with 15511 points.
隐藏点个数为： PointCloud with 97075 points.
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: 不支持请求的转换操作。 
